# 70 Cost-guided search on all train questions (A100)

Method "cost first, then accuracy": (1) a cost-guided search finds the cheapest correct query known for every train question; (2) the current model (`rl_v1_it8`) answers every train question 8 times; (3) preference pairs: chosen = the cheap query, rejected = the model's own costlier or wrong answer; (4) DPO from the current model; (5) RL from the DPO model, with the cheapest known query as the cost yardstick; (6) the model is chosen on validation by a fixed rule, then dev and mini-dev are scored once.

This is step 1. For every train question, the 7B model sees the cheapest correct query known so far (from the first pool, which your generated queries built), its cost in SQLite steps and its plan, and its earlier failed attempts; it proposes 4 cheaper rewrites per round, 3 rounds. Every rewrite is executed and kept only under the strict rule: same result as gold, no first-row pick that gold does not use, no constant found nowhere in the question, evidence or gold.

Output `runs/search_v1/cheap_targets_train.jsonl`: per question the cheapest usable query (the chosen answer of DPO, the cost yardstick of RL) and the first pool's own cheapest.

**Runtime:** A100, about 3 to 4 hours (the 300-question pilot took 5 minutes of generation). **Resumable:** after a disconnect, run the setup and config cells (and the staging cell if there is one) again, then the cell that was running: finished work is kept.

**Then:** send me a message; I check the headroom (decision D1) before step 2.

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
EXPORT = "sft_v1"                   # SFT export: its 4 held-out train databases are the validation set
CURRENT = "rl_v1_it8"               # our current model (RL, kept on validation): the start and the reference
SEARCH = "search_v1"                # 1. cost-guided search, all train questions -> cheap targets
OWN = "own_v1"                      # 2. the current model's own answers, 8 per train question
PAIRS = "pairs_v2"                  # 3. preference pairs (main)
DPO = "dpo_v2"                      # 4. DPO from CURRENT, 2 epochs: dpo_v2_e1 (after epoch 1) and dpo_v2
DPO_RUNS = [f"{DPO}_e1", DPO]
RL = "rl_v2"                        # 5. RL from the DPO model kept on validation: rl_v2_it8 and rl_v2
RL_RUNS = [f"{RL}_it8", RL]
# Ablations (DPO only, same settings as the kept DPO model): where the chosen and rejected answers come from.
ABLATIONS = {"gold": "--chosen gold",      # chosen = gold: the generated queries are not used at all
             "pool": "--chosen pool",      # chosen = the first pool's cheapest query: no search
             "off": "--rejected pool"}     # rejected = other models' pool queries, not the model's own
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
VAL_DBS = ",".join(json.load(open(f"{R}/{EXPORT}/export_meta.json"))["holdout_db_ids"])
def done(run):
    return os.path.exists(f"{R}/{run}/adapter_meta.json")
print("validation databases:", VAL_DBS)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

### Stage the train databases (30 GB, a few minutes; skipped when already staged)

In [ ]:
!effsql stage --split train
!effsql schema-cache --split train

### 1. Search (3 rounds over all train questions)

In [ ]:
!python -m effsql.gen.search --run {SEARCH} --model 7b

### 2. Headroom: training and validation questions, first pool alone and with the search

`pct_2x`: share of questions with a usable query at least 2x cheaper than gold; `pct_10pct`: at least 10% cheaper; `geomean_rel_cost`: geometric mean cost of the cheapest usable query relative to gold (1 = gold). Check (D1): the first pool alone already gives, on the training questions, 5.5% 2x cheaper, 20.1% 10% cheaper and 0.830 (validation: 2.9%, 17.3%, 0.915). The search should lower these clearly (the pilot: 0.88 to 0.80 on its 300 questions).

In [ ]:
s = json.load(open(f"{R}/{SEARCH}/search_summary_train.json"))
for part in ("train", "val"):
    f = s["final"][part]
    print(part, "first pool:", f["pool_only"])
    print(part, "with search:", f["with_search"], "| cheapest from:", f["by_source"])
for r in s["rounds"]:
    print("round", r["round"], {k: r[k] for k in ("pct_2x", "pct_10pct", "geomean_rel_cost", "n_new", "pct_new_correct", "n_questions_improved")})
for g in s["generation"]:
    print("generation round", g["round"], f'{g["n_active"]} questions, {g["elapsed_s"] / 60:.0f} min')

### Done: disconnect the runtime (an idle A100 still costs units)